# Jour 4 — Clustering et anomalies

## Notebook de synthèse

**Idée directrice :** en non supervisé, le résultat dépend de la représentation des données, de la distance utilisée et de l'hypothèse géométrique de la méthode.

### Objectifs
- Montrer pourquoi un cluster dépend de la représentation.
- Comprendre la PCA comme projection utile, pas comme vérité cachée.
- Comparer K-means, GMM et DBSCAN sur les mêmes données.
- Introduire la détection d'anomalies par densité locale et par isolement.
- Apprendre à conclure sans disposer d'une cible supervisée.

### Mode d'emploi
Pour chaque section :
1. Lire la **question**.
2. Exécuter le **code minimal**.
3. Compléter la **lecture du résultat**.
4. Rédiger une **conclusion courte** en langage métier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest

sns.set_theme(style="whitegrid", context="talk")
np.random.seed(42)

## 1. Représentation et distance

### Question
Si l'on change l'échelle d'une variable, est-ce que la notion de proximité reste la même ?

### À retenir
Une méthode fondée sur des distances peut changer fortement de comportement si une variable domine numériquement les autres.

In [ ]:
X1, y1 = make_blobs(
    n_samples=300,
    centers=[(-2, -2), (2, 2), (2, -2)],
    cluster_std=[1.0, 1.2, 0.8],
    random_state=42
)

X1_scaled_axis = X1.copy()
X1_scaled_axis[:, 1] *= 25

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(X1[:, 0], X1[:, 1], c=y1, cmap='viridis', s=30)
axes[0].set_title('Données initiales')
axes[0].set_xlabel('x1')
axes[0].set_ylabel('x2')

axes[1].scatter(X1_scaled_axis[:, 0], X1_scaled_axis[:, 1], c=y1, cmap='viridis', s=30)
axes[1].set_title('Même structure, axe x2 multiplié par 25')
axes[1].set_xlabel('x1')
axes[1].set_ylabel('x2 modifié')

plt.tight_layout()
plt.show()

In [ ]:
km_raw = KMeans(n_clusters=3, n_init=20, random_state=42)
labels_raw = km_raw.fit_predict(X1_scaled_axis)

scaler = StandardScaler()
X1_std = scaler.fit_transform(X1_scaled_axis)
km_std = KMeans(n_clusters=3, n_init=20, random_state=42)
labels_std = km_std.fit_predict(X1_std)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(X1_scaled_axis[:, 0], X1_scaled_axis[:, 1], c=labels_raw, cmap='tab10', s=30)
axes[0].set_title('K-means sans standardisation')

axes[1].scatter(X1_scaled_axis[:, 0], X1_scaled_axis[:, 1], c=labels_std, cmap='tab10', s=30)
axes[1].set_title('K-means après standardisation')

plt.tight_layout()
plt.show()

### Lecture du résultat
- Sans standardisation, la variable étirée peut dominer la distance.
- Après standardisation, on redonne un poids comparable aux variables.
- **Conclusion attendue :** un cluster n'est jamais indépendant de la représentation choisie.

## 2. PCA : projeter sans prétendre tout conserver

### Question
Que garde exactement une PCA lorsqu'on passe de plusieurs dimensions à deux axes ?

### À retenir
La PCA conserve de la variance, pas automatiquement du sens métier ni la meilleure séparation pour une tâche donnée.

In [ ]:
rng = np.random.RandomState(42)
base = rng.normal(size=(500, 3))
X2 = np.column_stack([
    2.5 * base[:, 0] + 0.3 * base[:, 1],
    2.0 * base[:, 0] - 0.4 * base[:, 1] + 0.2 * base[:, 2],
    0.5 * base[:, 0] + 1.7 * base[:, 1],
    0.2 * base[:, 0] + 0.1 * base[:, 1] + 2.2 * base[:, 2],
])

X2_std = StandardScaler().fit_transform(X2)
pca_full = PCA().fit(X2_std)
var_exp = pca_full.explained_variance_ratio_
var_cum = np.cumsum(var_exp)

pd.DataFrame({
    'composante': np.arange(1, len(var_exp) + 1),
    'variance_expliquee': var_exp,
    'variance_cumulee': var_cum
})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, len(var_exp) + 1), var_exp, alpha=0.7, label='Variance expliquée')
ax.plot(range(1, len(var_cum) + 1), var_cum, marker='o', color='crimson', label='Variance cumulée')
ax.set_xlabel('Composante principale')
ax.set_ylabel('Part de variance')
ax.set_title('Lecture de la variance expliquée')
ax.set_xticks(range(1, len(var_exp) + 1))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
pca_2 = PCA(n_components=2)
X2_pca = pca_2.fit_transform(X2_std)

plt.figure(figsize=(8, 6))
plt.scatter(X2_pca[:, 0], X2_pca[:, 1], s=25, alpha=0.7)
plt.title('Projection PCA en 2D')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

### Lecture du résultat
- La variance cumulée aide à décider combien d'axes garder.
- Une projection 2D est utile pour voir une structure, mais simplifie toujours le nuage initial.
- **Conclusion attendue :** une bonne compression géométrique n'est pas une preuve de pertinence métier.

## 3. K-means : inertie et compromis sur K

### Question
Comment K-means choisit-il ses groupes, et pourquoi le choix de K ne découle-t-il pas automatiquement d'un seul graphique ?

In [ ]:
X3, _ = make_blobs(
    n_samples=500,
    centers=4,
    cluster_std=[0.7, 0.9, 1.1, 0.6],
    random_state=42
)
X3_std = StandardScaler().fit_transform(X3)

results = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X3_std)
    results.append({
        'K': k,
        'inertie': km.inertia_,
        'silhouette': silhouette_score(X3_std, labels)
    })

res_k = pd.DataFrame(results)
res_k

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(res_k['K'], res_k['inertie'], marker='o')
axes[0].set_title('Méthode du coude')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertie')

axes[1].plot(res_k['K'], res_k['silhouette'], marker='o', color='darkgreen')
axes[1].set_title('Score de silhouette')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette moyenne')

plt.tight_layout()
plt.show()

In [ ]:
k_best = 4
km4 = KMeans(n_clusters=k_best, n_init=20, random_state=42)
labels4 = km4.fit_predict(X3_std)

plt.figure(figsize=(8, 6))
plt.scatter(X3_std[:, 0], X3_std[:, 1], c=labels4, cmap='tab10', s=30)
plt.scatter(km4.cluster_centers_[:, 0], km4.cluster_centers_[:, 1], c='black', s=180, marker='X')
plt.title(f'K-means avec K = {k_best}')
plt.tight_layout()
plt.show()

### Lecture du résultat
- L'inertie décroît presque toujours quand K augmente.
- La silhouette aide à juger compacité et séparation, mais ne remplace pas l'interprétation.
- **Conclusion attendue :** le “bon” K est un compromis entre structure géométrique, stabilité et utilité analytique.

## 4. Quand K-means échoue : les deux lunes

### Question
Pourquoi une méthode efficace sur des groupes compacts peut-elle échouer sur une structure courbe ?

In [ ]:
X4, y4 = make_moons(n_samples=400, noise=0.07, random_state=42)
X4_std = StandardScaler().fit_transform(X4)

km_moons = KMeans(n_clusters=2, n_init=20, random_state=42)
labels_km_moons = km_moons.fit_predict(X4_std)

db_moons = DBSCAN(eps=0.25, min_samples=5)
labels_db_moons = db_moons.fit_predict(X4_std)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(X4_std[:, 0], X4_std[:, 1], c=y4, cmap='coolwarm', s=25)
axes[0].set_title('Structure de référence')

axes[1].scatter(X4_std[:, 0], X4_std[:, 1], c=labels_km_moons, cmap='tab10', s=25)
axes[1].set_title('K-means sur les lunes')

axes[2].scatter(X4_std[:, 0], X4_std[:, 1], c=labels_db_moons, cmap='tab10', s=25)
axes[2].set_title('DBSCAN sur les lunes')

plt.tight_layout()
plt.show()

### Lecture du résultat
- K-means favorise des régions autour de centres.
- DBSCAN peut mieux suivre une structure courbe car il raisonne par densité connectée.
- **Conclusion attendue :** un algorithme ne “voit” pas les formes comme un humain ; il optimise son propre critère.

## 5. Comparer plusieurs familles sur le même jeu de données

### Question
Que gagne-t-on à confronter plusieurs hypothèses de structure sur les mêmes points ?

In [ ]:
X5, _ = make_blobs(
    n_samples=500,
    centers=[(-3, -1), (0, 2.5), (3.5, -0.5)],
    cluster_std=[0.9, 1.3, 0.8],
    random_state=42
)
X5 = np.vstack([X5, np.array([[6, 6], [6.5, 5.8], [-6, 4]])])
X5_std = StandardScaler().fit_transform(X5)

models = {
    'KMeans': KMeans(n_clusters=3, n_init=20, random_state=42).fit_predict(X5_std),
    'GMM': GaussianMixture(n_components=3, random_state=42).fit(X5_std).predict(X5_std),
    'DBSCAN': DBSCAN(eps=0.30, min_samples=6).fit_predict(X5_std),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, labels) in zip(axes, models.items()):
    ax.scatter(X5_std[:, 0], X5_std[:, 1], c=labels, cmap='tab10', s=30)
    ax.set_title(name)
plt.tight_layout()
plt.show()

### Lecture du résultat
- K-means produit des groupes compacts autour de centres.
- GMM autorise une lecture probabiliste et des formes elliptiques.
- DBSCAN peut isoler du bruit et s'affranchir du choix explicite de K.
- **Conclusion attendue :** changer de méthode revient à changer d'hypothèse sur la structure des données.

## 6. Anomalies : densité locale contre isolement

### Question
Deux méthodes différentes signalent-elles les mêmes points atypiques ?

In [ ]:
X6, _ = make_blobs(n_samples=350, centers=[(-2, -2), (2, 2)], cluster_std=[0.8, 1.0], random_state=42)
outliers = np.array([[6, 6], [7, 6.5], [-6, 5], [0, 7], [8, -3]])
X6 = np.vstack([X6, outliers])
X6_std = StandardScaler().fit_transform(X6)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)
lof_labels = lof.fit_predict(X6_std)
lof_scores = -lof.negative_outlier_factor_

iso = IsolationForest(contamination=0.02, random_state=42)
iso.fit(X6_std)
iso_labels = iso.predict(X6_std)
iso_scores = -iso.score_samples(X6_std)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].scatter(X6_std[:, 0], X6_std[:, 1], c=lof_scores, cmap='magma', s=35)
axes[0].set_title("LOF : score d'anomalie")

axes[1].scatter(X6_std[:, 0], X6_std[:, 1], c=iso_scores, cmap='magma', s=35)
axes[1].set_title("Isolation Forest : score d'anomalie")

plt.tight_layout()
plt.show()

In [ ]:
out_df = pd.DataFrame({
    'lof_anomalie': lof_labels == -1,
    'score_lof': lof_scores,
    'iso_anomalie': iso_labels == -1,
    'score_iso': iso_scores
})
out_df.sort_values(['score_lof', 'score_iso'], ascending=False).head(10)

### Lecture du résultat
- LOF compare une densité locale à celle du voisinage.
- Isolation Forest récompense les points faciles à isoler.
- Les deux méthodes peuvent se recouper sans être identiques.
- **Conclusion attendue :** une anomalie dépend de la définition opérationnelle de l'atypique.

## 7. Évaluer sans labels : une petite grille de lecture

### Question
Comment conclure proprement lorsqu'on n'a pas de vérité terrain complète ?

In [ ]:
summary = pd.DataFrame({
    'Question': [
        'Les groupes sont-ils stables ?',
        'Les groupes sont-ils séparés ?',
        'Existe-t-il du bruit interprétable ?',
        'Les anomalies sont-elles actionnables ?',
        'Le résultat dépend-il trop des paramètres ?'
    ],
    'Diagnostic possible': [
        'Changer la graine, sous-échantillonner, comparer plusieurs exécutions',
        'Silhouette, visualisation, distances intra/inter',
        'DBSCAN/HDBSCAN, inspection de cas limites',
        'Inspection métier, coût d’investigation, volume d’alertes',
        'Analyse de sensibilité sur eps, K, contamination, n_neighbors'
    ]
})
summary

### Lecture du résultat
- En non supervisé, une métrique unique ne suffit pas.
- Il faut articuler géométrie, stabilité, cas limites et utilité métier.
- **Conclusion attendue :** un résultat non supervisé est un diagnostic argumenté, pas une vérité révélée.

## 8. Mini synthèse à retenir

### Les 5 messages de la journée
1. Un cluster dépend toujours de la représentation des données.
2. La PCA projette utilement, mais elle simplifie toujours le problème.
3. K-means, GMM et DBSCAN n'expriment pas la même hypothèse géométrique.
4. Une anomalie peut être définie par densité locale, isolement ou frontière du normal.
5. Sans labels, on conclut par convergence d'indices, pas avec une seule métrique.

### Question de restitution
Rédiger en 8 à 10 lignes :
- la méthode que vous choisiriez pour des groupes compacts ;
- la méthode que vous choisiriez pour des formes courbes ;
- la méthode d'anomalie que vous testeriez en premier ;
- ce que vous vérifieriez avant toute conclusion métier.